# Statistical & Meteorological Analysis

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from itertools import combinations

from scipy.stats import (
    kruskal,
    mannwhitneyu,
    spearmanr,
    chi2_contingency
)

import statsmodels.formula.api as smf

from statsmodels.stats.multitest import (
    multipletests
)

from statsmodels.stats.outliers_influence import (
    variance_inflation_factor
)

from statsmodels.stats.stattools import (
    durbin_watson
)

from statsmodels.stats.diagnostic import (
    acorr_ljungbox
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

sns.set_theme(style="whitegrid")

In [2]:
PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

REPORTS_DIR = PROJECT_ROOT / "reports"
TABLES_DIR = (
    REPORTS_DIR /
    "tables" /
    "phase6_statistics"
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
hourly = pd.read_csv(
    PROCESSED_DIR /
    "air_quality_features.csv",
    parse_dates=['time'],
    index_col='time'
)

daily = pd.read_csv(
    PROCESSED_DIR /
    "air_quality_daily.csv",
    parse_dates=['time'],
    index_col='time'
)

In [4]:
print("Hourly:", hourly.shape)
print("Daily:", daily.shape)

print(
    "Hourly missing:",
    hourly.isna().sum().sum()
)

print(
    "Daily missing:",
    daily.isna().sum().sum()
)

Hourly: (23352, 43)
Daily: (973, 30)
Hourly missing: 0
Daily missing: 0


In [5]:
season_order = [
    'Winter',
    'Pre-Monsoon',
    'Monsoon',
    'Post-Monsoon'
]

day_order = [
    'Monday',
    'Tuesday',
    'Wednesday',
    'Thursday',
    'Friday',
    'Saturday',
    'Sunday'
]

aqi_order = [
    'Good',
    'Moderate',
    'Unhealthy for Sensitive Groups',
    'Unhealthy',
    'Very Unhealthy',
    'Hazardous'
]

In [6]:
hourly['season'] = pd.Categorical(
    hourly['season'],
    categories=season_order,
    ordered=True
)

daily['season'] = pd.Categorical(
    daily['season'],
    categories=season_order,
    ordered=True
)

daily['particle_aqi_category'] = pd.Categorical(
    daily['particle_aqi_category'],
    categories=aqi_order,
    ordered=True
)

## Seasonal differences in PM2.5

In [7]:
#Daily PM2.5 by season
season_daily_summary = (
    daily
    .groupby(
        'season',
        observed=False
    )['pm25_mean']
    .agg([
        'count',
        'mean',
        'median',
        'std'
    ])
)

season_daily_summary

,count,mean,median,std
season,,,,
Winter,271,51.067743,48.816667,17.885526
Pre-Monsoon,245,33.942330,31.683333,13.469735
Monsoon,274,20.006280,18.058333,9.323372
Post-Monsoon,183,35.312819,29.458333,19.333413


### Kruskal–Wallis test(Because PM2.5 is strongly right-skewed, using a non-parametric test.)
H₀:
PM2.5 distributions are equal across seasons.

H₁:
At least one season differs.

In [9]:
season_groups = [
    daily.loc[
        daily['season'] == season,
        'pm25_mean'
    ].dropna()
    for season in season_order
]

kw_season = kruskal(
    *season_groups
)

print(
    "Kruskal-Wallis H:",
    kw_season.statistic
)

print(
    "p-value:",
    kw_season.pvalue
)

Kruskal-Wallis H: 420.0755926585559
p-value: 9.916947881866245e-91


In [10]:
n = sum(
    len(group)
    for group in season_groups
)

k = len(
    season_groups
)

epsilon_squared = (
    (
        kw_season.statistic
        - k
        + 1
    )
    /
    (n - k)
)

print(
    "Epsilon-squared:",
    epsilon_squared
)

Epsilon-squared: 0.43041856827508346


In [11]:
#Pairwise test helper
def pairwise_mannwhitney(
    df,
    group_column,
    value_column,
    group_order
):

    results = []

    for group1, group2 in combinations(
        group_order,
        2
    ):

        x = df.loc[
            df[group_column] == group1,
            value_column
        ].dropna()

        y = df.loc[
            df[group_column] == group2,
            value_column
        ].dropna()

        test = mannwhitneyu(
            x,
            y,
            alternative='two-sided'
        )

        n1 = len(x)
        n2 = len(y)

        # Rank-biserial correlation
        effect_size = (
            2 * test.statistic
            / (n1 * n2)
            - 1
        )

        results.append({
            'group_1': group1,
            'group_2': group2,

            'n_1': n1,
            'n_2': n2,

            'median_1': x.median(),
            'median_2': y.median(),

            'U': test.statistic,
            'p_raw': test.pvalue,

            'rank_biserial':
                effect_size
        })

    results = pd.DataFrame(
        results
    )

    reject, p_adjusted, _, _ = (
        multipletests(
            results['p_raw'],
            method='holm'
        )
    )

    results[
        'p_holm'
    ] = p_adjusted

    results[
        'significant_holm'
    ] = reject

    return results

In [12]:
#Seasonal pairwise comparisons
season_pairwise = (
    pairwise_mannwhitney(
        daily,
        'season',
        'pm25_mean',
        season_order
    )
)

season_pairwise.round(4)

,group_1,group_2,n_1,n_2,median_1,median_2,U,p_raw,rank_biserial,p_holm,significant_holm
0,Winter,Pre-Monsoon,271,245,48.8167,31.6833,52314.0,0.0000,0.5758,0.0000,True
1,Winter,Monsoon,271,274,48.8167,18.0583,70455.0,0.0000,0.8977,0.0000,True
2,Winter,Post-Monsoon,271,183,48.8167,29.4583,37250.0,0.0000,0.5022,0.0000,True
3,Pre-Monsoon,Monsoon,245,274,31.6833,18.0583,55822.0,0.0000,0.6631,0.0000,True
4,Pre-Monsoon,Post-Monsoon,245,183,31.6833,29.4583,23144.5,0.5661,0.0324,0.5661,False
5,Monsoon,Post-Monsoon,274,183,18.0583,29.4583,11270.0,0.0000,-0.5505,0.0000,True


## Weekday vs weekend

In [13]:
daily['day_type'] = np.where(
    daily.index.dayofweek >= 5,
    'Weekend',
    'Weekday'
)

In [14]:
weekday_summary = (
    daily
    .groupby('day_type')['pm25_mean']
    .agg([
        'count',
        'mean',
        'median',
        'std'
    ])
)

weekday_summary

,count,mean,median,std
day_type,,,,
Weekday,695,35.275432,30.483333,19.413950
Weekend,278,34.470399,31.860417,18.279241


In [15]:
#Mann–Whitney test
weekday_values = (
    daily.loc[
        daily['day_type']
        == 'Weekday',
        'pm25_mean'
    ]
)

weekend_values = (
    daily.loc[
        daily['day_type']
        == 'Weekend',
        'pm25_mean'
    ]
)

weekday_test = mannwhitneyu(
    weekday_values,
    weekend_values,
    alternative='two-sided'
)

weekday_test

MannwhitneyuResult(statistic=np.float64(97720.0), pvalue=np.float64(0.778377598334322))

In [16]:
n_weekday = len(
    weekday_values
)

n_weekend = len(
    weekend_values
)

weekday_rank_biserial = (
    2 * weekday_test.statistic
    /
    (
        n_weekday
        * n_weekend
    )
    - 1
)

print(
    "Rank-biserial effect:",
    weekday_rank_biserial
)

Rank-biserial effect: 0.011541845660162586


In [17]:
#Day-of-week differences
daily['day_name'] = (
    daily.index.day_name()
)

In [18]:
dow_groups = [
    daily.loc[
        daily['day_name'] == day,
        'pm25_mean'
    ]
    for day in day_order
]

kw_dow = kruskal(
    *dow_groups
)

print(
    "Day-of-week Kruskal-Wallis H:",
    kw_dow.statistic
)

print(
    "p-value:",
    kw_dow.pvalue
)

Day-of-week Kruskal-Wallis H: 0.9264400394017257
p-value: 0.9882477077791485


In [19]:
n_dow = sum(
    len(group)
    for group in dow_groups
)

k_dow = len(
    dow_groups
)

dow_epsilon_squared = (
    (
        kw_dow.statistic
        - k_dow
        + 1
    )
    /
    (
        n_dow
        - k_dow
    )
)

print(
    "Day-of-week epsilon²:",
    dow_epsilon_squared
)

Day-of-week epsilon²: -0.005252132464387447


## Season vs AQI category

In [20]:
season_aqi_table = pd.crosstab(
    daily['season'],
    daily[
        'particle_aqi_category'
    ]
)

In [21]:
season_aqi_table = (
    season_aqi_table.loc[
        :,
        season_aqi_table.sum(
            axis=0
        ) > 0
    ]
)

season_aqi_table

particle_aqi_category,Good,Moderate,Unhealthy for Sensitive Groups,Unhealthy
season,,,,
Winter,2,54,110,105
Pre-Monsoon,0,154,76,15
Monsoon,17,236,21,0
Post-Monsoon,2,111,45,25


In [22]:
#Chi-square test
chi2, p_value, dof, expected = (
    chi2_contingency(
        season_aqi_table
    )
)

print("Chi-square:", chi2)
print("Degrees of freedom:", dof)
print("p-value:", p_value)

Chi-square: 354.9745398814661
Degrees of freedom: 9
p-value: 5.411269206149704e-71


In [23]:
#Cramér's V
n = (
    season_aqi_table
    .to_numpy()
    .sum()
)

rows, cols = (
    season_aqi_table.shape
)

cramers_v = np.sqrt(
    chi2
    /
    (
        n
        * min(
            rows - 1,
            cols - 1
        )
    )
)

print(
    "Cramér's V:",
    cramers_v
)

Cramér's V: 0.3487237730853125


## Meteorological correlations

In [24]:
pollutants_daily = [
    'pm25_mean',
    'pm10_mean',
    'no2_mean',
    'so2_mean',
    'co_mean'
]

In [25]:
weather_daily = [
    'temperature_mean',
    'humidity_mean',
    'rain_total',
    'wind_speed_mean',
    'pressure_mean',
    'dew_point_mean'
]

## Spearman inference table

In [26]:
correlation_results = []

for pollutant in pollutants_daily:

    for weather_var in weather_daily:

        rho, p = spearmanr(
            daily[pollutant],
            daily[weather_var],
            nan_policy='omit'
        )

        correlation_results.append({
            'pollutant': pollutant,
            'weather_variable':
                weather_var,

            'spearman_rho':
                rho,

            'p_raw':
                p
        })


correlation_results = pd.DataFrame(
    correlation_results
)

In [28]:
#Benjamini–Hochberg FDR correction
reject, p_fdr, _, _ = (
    multipletests(
        correlation_results[
            'p_raw'
        ],
        method='fdr_bh'
    )
)

correlation_results[
    'p_fdr'
] = p_fdr

correlation_results[
    'significant_fdr'
] = reject

In [29]:
correlation_results = (
    correlation_results
    .sort_values(
        by='spearman_rho'
    )
)

correlation_results.round(4)

,pollutant,weather_variable,spearman_rho,p_raw,p_fdr,significant_fdr
24,co_mean,temperature_mean,-0.7594,0.000,0.000,True
18,so2_mean,temperature_mean,-0.6660,0.000,0.000,True
29,co_mean,dew_point_mean,-0.6361,0.000,0.000,True
0,pm25_mean,temperature_mean,-0.6222,0.000,0.000,True
26,co_mean,rain_total,-0.5982,0.000,0.000,True
23,so2_mean,dew_point_mean,-0.5962,0.000,0.000,True
5,pm25_mean,dew_point_mean,-0.5831,0.000,0.000,True
6,pm10_mean,temperature_mean,-0.5653,0.000,0.000,True
12,no2_mean,temperature_mean,-0.5575,0.000,0.000,True
20,so2_mean,rain_total,-0.5537,0.000,0.000,True


## Rainfall comparison

In [32]:
#dry/wet day
daily['rain_day'] = np.where(
    daily['rain_total'] > 0,
    'Wet',
    'Dry'
)

In [33]:
rain_day_summary = (
    daily
    .groupby('rain_day')['pm25_mean']
    .agg([
        'count',
        'mean',
        'median',
        'std'
    ])
)

rain_day_summary

,count,mean,median,std
rain_day,,,,
Dry,390,43.375043,40.120833,18.712280
Wet,583,29.473292,24.895833,17.235173


## Mann–Whitney test

In [34]:
dry_days = daily.loc[
    daily['rain_day'] == 'Dry',
    'pm25_mean'
]

wet_days = daily.loc[
    daily['rain_day'] == 'Wet',
    'pm25_mean'
]

rain_test = mannwhitneyu(
    dry_days,
    wet_days,
    alternative='two-sided'
)

rain_test

MannwhitneyuResult(statistic=np.float64(165991.5), pvalue=np.float64(4.1847355993173564e-34))

In [35]:
rain_effect_size = (
    2 * rain_test.statistic
    /
    (
        len(dry_days)
        * len(wet_days)
    )
    - 1
)

print(
    "Rain-day rank-biserial effect:",
    rain_effect_size
)

Rain-day rank-biserial effect: 0.4601002770814091


## Multicollinearity

In [36]:
#Prepare predictor dataset
vif_data = hourly[
    [
        'temperature',
        'humidity',
        'wind_speed',
        'pressure',
        'dew_point'
    ]
].copy()

In [37]:
#standardized
vif_standardized = (
    vif_data
    - vif_data.mean()
) / vif_data.std()

In [38]:
#vif
vif_table = pd.DataFrame({
    'variable':
        vif_standardized.columns,

    'VIF': [
        variance_inflation_factor(
            vif_standardized.values,
            i
        )
        for i in range(
            vif_standardized.shape[1]
        )
    ]
})

vif_table

,variable,VIF
0,temperature,24.445955
1,humidity,21.367398
2,wind_speed,1.338783
3,pressure,1.220296
4,dew_point,32.777571


## Time-aware regression dataset

In [39]:
model_df = hourly.copy()

In [40]:
model_df['log_pm25'] = (
    np.log1p(
        model_df['pm25']
    )
)

In [41]:
#lag-1 PM2.5
model_df['pm25_lag1'] = (
    model_df['pm25']
    .shift(1)
)

In [42]:
#Rain transformation
model_df['log_rain'] = (
    np.log1p(
        model_df['rain']
    )
)

In [43]:
continuous_predictors = [
    'temperature',
    'humidity',
    'wind_speed',
    'log_rain',
    'pressure',
    'pm25_lag1'
]

In [44]:
for column in continuous_predictors:

    mean = model_df[
        column
    ].mean()

    std = model_df[
        column
    ].std()

    model_df[
        f'z_{column}'
    ] = (
        model_df[column]
        - mean
    ) / std

In [45]:
model_df = (
    model_df
    .dropna()
    .copy()
)

## Hourly OLS + HAC/Newey–West

In [46]:
#regression
formula = """
log_pm25 ~
z_temperature +
z_humidity +
z_wind_speed +
z_log_rain +
z_pressure +
wind_dir_sin +
wind_dir_cos +
z_pm25_lag1 +
C(hour) +
C(month)
"""

In [47]:
hourly_model = smf.ols(
    formula=formula,
    data=model_df
).fit(
    cov_type='HAC',
    cov_kwds={
        'maxlags': 24
    }
)

print(
    hourly_model.summary()
)

                            OLS Regression Results                            
Dep. Variable:               log_pm25   R-squared:                       0.828
Model:                            OLS   Adj. R-squared:                  0.828
Method:                 Least Squares   F-statistic:                     242.3
Date:                Sun, 27 Sep 2026   Prob (F-statistic):               0.00
Time:                        18:22:04   Log-Likelihood:                -2275.3
No. Observations:               23351   AIC:                             4637.
Df Residuals:                   23308   BIC:                             4983.
Df Model:                          42                                         
Covariance Type:                  HAC                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          3.4059      0.039     86.

In [48]:
#coefficients
weather_terms = [
    'z_temperature',
    'z_humidity',
    'z_wind_speed',
    'z_log_rain',
    'z_pressure',
    'wind_dir_sin',
    'wind_dir_cos',
    'z_pm25_lag1'
]

In [49]:
hourly_coef_table = pd.DataFrame({
    'coefficient':
        hourly_model.params[
            weather_terms
        ],

    'std_error':
        hourly_model.bse[
            weather_terms
        ],

    'p_value':
        hourly_model.pvalues[
            weather_terms
        ],

    'ci_lower':
        hourly_model.conf_int()
        .loc[
            weather_terms,
            0
        ],

    'ci_upper':
        hourly_model.conf_int()
        .loc[
            weather_terms,
            1
        ]
})

hourly_coef_table

,coefficient,std_error,p_value,ci_lower,ci_upper
z_temperature,-0.001790,0.014438,9.013524e-01,-0.030088,0.026509
z_humidity,-0.000952,0.012755,9.404889e-01,-0.025952,0.024047
z_wind_speed,-0.018312,0.005069,3.032984e-04,-0.028248,-0.008377
z_log_rain,-0.043855,0.009143,1.612795e-06,-0.061774,-0.025935
z_pressure,0.043350,0.008560,4.096580e-07,0.026573,0.060127
wind_dir_sin,-0.058379,0.009701,1.769197e-09,-0.077393,-0.039365
wind_dir_cos,-0.001125,0.006888,8.702801e-01,-0.014625,0.012375
z_pm25_lag1,0.522539,0.017357,4.181031e-199,0.488519,0.556559


In [50]:
#coefficients to approximate percentage changes
hourly_coef_table[
    'approx_percent_change'
] = (
    np.exp(
        hourly_coef_table[
            'coefficient'
        ]
    )
    - 1
) * 100

In [51]:
hourly_coef_table.round(4)

,coefficient,std_error,p_value,ci_lower,ci_upper,approx_percent_change
z_temperature,-0.0018,0.0144,0.9014,-0.0301,0.0265,-0.1788
z_humidity,-0.0010,0.0128,0.9405,-0.0260,0.0240,-0.0952
z_wind_speed,-0.0183,0.0051,0.0003,-0.0282,-0.0084,-1.8146
z_log_rain,-0.0439,0.0091,0.0000,-0.0618,-0.0259,-4.2907
z_pressure,0.0434,0.0086,0.0000,0.0266,0.0601,4.4304
wind_dir_sin,-0.0584,0.0097,0.0000,-0.0774,-0.0394,-5.6707
wind_dir_cos,-0.0011,0.0069,0.8703,-0.0146,0.0124,-0.1124
z_pm25_lag1,0.5225,0.0174,0.0000,0.4885,0.5566,68.6304


In [52]:
#Model explanatory power
print(
    "R²:",
    hourly_model.rsquared
)

print(
    "Adjusted R²:",
    hourly_model.rsquared_adj
)

print(
    "Number of observations:",
    int(hourly_model.nobs)
)

R²: 0.828436784827423
Adjusted R²: 0.8281276353921541
Number of observations: 23351


## Residual temporal dependence

In [53]:
#Durbin–Watson
dw = durbin_watson(
    hourly_model.resid
)

print(
    "Durbin-Watson:",
    dw
)

Durbin-Watson: 0.4135434402305301


In [54]:
#Ljung–Box residual test
residual_ljung = acorr_ljungbox(
    hourly_model.resid,
    lags=[1, 6, 12, 24],
    return_df=True
)

residual_ljung

,lb_stat,lb_pvalue
1,14691.281526,0.0
6,53645.917093,0.0
12,73217.734298,0.0
24,114961.020081,0.0


## Daily regression as a robustness check

In [55]:
daily_model_df = daily.copy()

In [56]:
daily_model_df['log_pm25'] = (
    np.log1p(
        daily_model_df[
            'pm25_mean'
        ]
    )
)

daily_model_df[
    'pm25_lag1'
] = (
    daily_model_df[
        'pm25_mean'
    ]
    .shift(1)
)

daily_model_df[
    'log_rain'
] = np.log1p(
    daily_model_df[
        'rain_total'
    ]
)

In [57]:
daily_wind_rad = np.deg2rad(
    daily_model_df[
        'wind_direction_mean'
    ]
)

daily_model_df[
    'wind_dir_sin'
] = np.sin(
    daily_wind_rad
)

daily_model_df[
    'wind_dir_cos'
] = np.cos(
    daily_wind_rad
)

In [58]:
#Standardize daily predictors
daily_predictors = [
    'temperature_mean',
    'humidity_mean',
    'wind_speed_mean',
    'log_rain',
    'pressure_mean',
    'pm25_lag1'
]

In [59]:
for column in daily_predictors:

    daily_model_df[
        f'z_{column}'
    ] = (
        daily_model_df[column]
        - daily_model_df[column].mean()
    ) / daily_model_df[column].std()

In [60]:
daily_model_df = (
    daily_model_df
    .dropna()
    .copy()
)

In [61]:
#Daily regression
daily_formula = """
log_pm25 ~
z_temperature_mean +
z_humidity_mean +
z_wind_speed_mean +
z_log_rain +
z_pressure_mean +
wind_dir_sin +
wind_dir_cos +
z_pm25_lag1 +
C(month)
"""

In [62]:
daily_model = smf.ols(
    formula=daily_formula,
    data=daily_model_df
).fit(
    cov_type='HAC',
    cov_kwds={
        'maxlags': 7
    }
)

print(
    daily_model.summary()
)

                            OLS Regression Results                            
Dep. Variable:               log_pm25   R-squared:                       0.785
Model:                            OLS   Adj. R-squared:                  0.781
Method:                 Least Squares   F-statistic:                     150.9
Date:                Sun, 27 Sep 2026   Prob (F-statistic):          3.10e-271
Time:                        18:24:33   Log-Likelihood:                -61.377
No. Observations:                 972   AIC:                             162.8
Df Residuals:                     952   BIC:                             260.3
Df Model:                          19                                         
Covariance Type:                  HAC                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              3.4683      0

In [64]:
#Extract daily coefficients
daily_terms = [
    'z_temperature_mean',
    'z_humidity_mean',
    'z_wind_speed_mean',
    'z_log_rain',
    'z_pressure_mean',
    'wind_dir_sin',
    'wind_dir_cos',
    'z_pm25_lag1'
]

In [65]:
daily_coef_table = pd.DataFrame({
    'coefficient':
        daily_model.params[
            daily_terms
        ],

    'std_error':
        daily_model.bse[
            daily_terms
        ],

    'p_value':
        daily_model.pvalues[
            daily_terms
        ],

    'ci_lower':
        daily_model.conf_int()
        .loc[
            daily_terms,
            0
        ],

    'ci_upper':
        daily_model.conf_int()
        .loc[
            daily_terms,
            1
        ]
})

In [66]:
daily_coef_table[
    'approx_percent_change'
] = (
    np.exp(
        daily_coef_table[
            'coefficient'
        ]
    )
    - 1
) * 100

daily_coef_table.round(4)

,coefficient,std_error,p_value,ci_lower,ci_upper,approx_percent_change
z_temperature_mean,0.0245,0.0321,0.4452,-0.0384,0.0874,2.4811
z_humidity_mean,0.0530,0.0227,0.0194,0.0086,0.0975,5.4455
z_wind_speed_mean,-0.0427,0.0210,0.0426,-0.0839,-0.0014,-4.1756
z_log_rain,-0.0903,0.0234,0.0001,-0.1362,-0.0445,-8.6366
z_pressure_mean,0.0349,0.0127,0.0059,0.0100,0.0597,3.5473
wind_dir_sin,-0.0771,0.0234,0.0010,-0.1229,-0.0312,-7.4181
wind_dir_cos,-0.0797,0.0344,0.0205,-0.1471,-0.0123,-7.6574
z_pm25_lag1,0.4031,0.0202,0.0000,0.3636,0.4427,49.6528


## Optional advanced analysis: quantile regression

In [67]:
quantile_formula = """
log_pm25 ~
z_temperature_mean +
z_humidity_mean +
z_wind_speed_mean +
z_log_rain +
z_pressure_mean +
wind_dir_sin +
wind_dir_cos +
z_pm25_lag1 +
C(month)
"""

In [68]:
quantiles = [
    0.50,
    0.75,
    0.90
]

quantile_results = []

for q in quantiles:

    model = smf.quantreg(
        quantile_formula,
        daily_model_df
    ).fit(
        q=q,
        max_iter=5000
    )

    for term in daily_terms:

        quantile_results.append({
            'quantile': q,
            'term': term,

            'coefficient':
                model.params[
                    term
                ],

            'p_value':
                model.pvalues[
                    term
                ]
        })


quantile_results = pd.DataFrame(
    quantile_results
)

In [69]:
quantile_coef_table = (
    quantile_results
    .pivot(
        index='term',
        columns='quantile',
        values='coefficient'
    )
)

quantile_coef_table.round(4)

quantile,0.50,0.75,0.90
term,,,
wind_dir_cos,-0.0687,-0.0645,-0.0647
wind_dir_sin,-0.0522,-0.0444,-0.0785
z_humidity_mean,0.0709,0.0755,0.0505
z_log_rain,-0.0679,-0.0448,-0.0340
z_pm25_lag1,0.3897,0.3901,0.3633
z_pressure_mean,0.0367,0.0225,0.0001
z_temperature_mean,0.0303,0.0112,-0.0310
z_wind_speed_mean,-0.0014,-0.0129,-0.0238


Quantile-regression inference here is exploratory and is not corrected for the same temporal dependence as the primary HAC regressions. The HAC models remain the primary inferential models.

In [70]:
season_daily_summary.to_csv(
    TABLES_DIR /
    "seasonal_pm25_summary.csv"
)

season_pairwise.to_csv(
    TABLES_DIR /
    "season_pairwise_tests.csv",
    index=False
)

weekday_summary.to_csv(
    TABLES_DIR /
    "weekday_weekend_summary.csv"
)

season_aqi_table.to_csv(
    TABLES_DIR /
    "season_aqi_contingency.csv"
)

correlation_results.to_csv(
    TABLES_DIR /
    "meteorological_spearman_tests.csv",
    index=False
)

rain_day_summary.to_csv(
    TABLES_DIR /
    "rain_pm25_summary.csv"
)

vif_table.to_csv(
    TABLES_DIR /
    "meteorological_vif.csv",
    index=False
)

hourly_coef_table.to_csv(
    TABLES_DIR /
    "hourly_hac_regression.csv"
)

daily_coef_table.to_csv(
    TABLES_DIR /
    "daily_hac_regression.csv"
)

quantile_coef_table.to_csv(
    TABLES_DIR /
    "quantile_regression_coefficients.csv"
)

In [72]:
print(
    season_daily_summary.round(2)
)

print(
    season_pairwise.round(4)
)

print(
    "Season epsilon²:",
    epsilon_squared
)

print(
    weekday_summary.round(2)
)

print(
    weekday_test
)

print(
    "Weekday effect:",
    weekday_rank_biserial
)

print(
    "DOW:",
    kw_dow,
    dow_epsilon_squared
)

print(
    season_aqi_table
)

print(
    "Chi-square:",
    chi2,
    p_value,
    cramers_v
)

print(
    correlation_results.round(4)
)

print(
    rain_day_summary.round(2)
)

print(
    "Rain:",
    rain_test,
    rain_effect_size
)

print(
    vif_table.round(2)
)

print(
    hourly_coef_table.round(4)
)

print(
    daily_coef_table.round(4)
)

print(
    "Hourly adjusted R2:",
    hourly_model.rsquared_adj
)

print(
    "Daily adjusted R2:",
    daily_model.rsquared_adj
)

print(
    residual_ljung
)

              count   mean  median    std
season                                   
Winter          271  51.07   48.82  17.89
Pre-Monsoon     245  33.94   31.68  13.47
Monsoon         274  20.01   18.06   9.32
Post-Monsoon    183  35.31   29.46  19.33
       group_1       group_2  n_1  n_2  median_1  median_2        U   p_raw  \
0       Winter   Pre-Monsoon  271  245   48.8167   31.6833  52314.0  0.0000   
1       Winter       Monsoon  271  274   48.8167   18.0583  70455.0  0.0000   
2       Winter  Post-Monsoon  271  183   48.8167   29.4583  37250.0  0.0000   
3  Pre-Monsoon       Monsoon  245  274   31.6833   18.0583  55822.0  0.0000   
4  Pre-Monsoon  Post-Monsoon  245  183   31.6833   29.4583  23144.5  0.5661   
5      Monsoon  Post-Monsoon  274  183   18.0583   29.4583  11270.0  0.0000   

   rank_biserial  p_holm  significant_holm  
0         0.5758  0.0000              True  
1         0.8977  0.0000              True  
2         0.5022  0.0000              True  
3         0.66

## Phase 6 Summary: Statistical and Meteorological Analysis

Formal statistical analysis confirmed that climatological season is a major source of variation in Kathmandu PM2.5 concentrations. Daily PM2.5 differed substantially across seasons, with a Kruskal–Wallis statistic of approximately 420.1 and a large epsilon-squared effect size of approximately 0.43. Winter had the highest median daily PM2.5 concentration, while monsoon had the lowest. Holm-adjusted pairwise comparisons indicated statistically distinguishable distributions for all seasonal pairs except pre-monsoon and post-monsoon, whose distributions were similar.

In contrast, weekday–weekend and day-of-week differences were negligible. The weekday–weekend comparison was not statistically significant and had a near-zero rank-biserial effect size, while the seven-day comparison similarly provided no evidence of meaningful day-of-week variation.

Particulate AQI category was strongly associated with climatological season. Winter contained a much greater frequency of Unhealthy for Sensitive Groups and Unhealthy days, whereas monsoon was dominated by Moderate conditions. The season-by-AQI association was substantial, with Cramér's V of approximately 0.35.

Unadjusted meteorological correlations showed strong seasonal structure. Daily PM2.5 was negatively associated with temperature, dew point, rainfall, and humidity, while positive associations were observed with wind speed and surface pressure. However, multivariable regression demonstrated that several of these unadjusted relationships changed after temporal and meteorological adjustment.

The strong negative unadjusted association between temperature and PM2.5 largely disappeared after controlling for month, hour, recent pollution, and other meteorological variables, indicating that much of the raw temperature relationship was associated with seasonal confounding. Wind speed exhibited the opposite pattern: although its unadjusted correlation with PM2.5 was positive, adjusted hourly and daily models produced negative coefficients. This sign reversal illustrates the importance of accounting for seasonality, wind direction, and other covariates when interpreting meteorological relationships.

Rainfall remained negatively associated with PM2.5 in both unadjusted and adjusted analyses. Dry days had substantially higher PM2.5 than wet days, and log-transformed rainfall retained a negative coefficient in both hourly and daily HAC regressions. Surface pressure showed a consistently positive adjusted association with PM2.5.

Previous PM2.5 concentration remained one of the strongest predictors in both regression models, reinforcing the substantial temporal persistence identified during the time-series analysis. The hourly and daily models explained approximately 83% and 78% of observed variation, respectively, although substantial residual serial correlation remained. HAC/Newey–West standard errors were therefore used to provide inference robust to heteroskedasticity and temporal dependence.

Overall, the results demonstrate that meteorological associations cannot be interpreted reliably from simple correlations alone. Temporal persistence, seasonality, and covariance among weather variables materially affect the apparent relationships between meteorological conditions and air pollution.

In [73]:
hourly_model_48 = smf.ols(
    formula=formula,
    data=model_df
).fit(
    cov_type='HAC',
    cov_kwds={
        'maxlags': 48
    }
)

hourly_model_168 = smf.ols(
    formula=formula,
    data=model_df
).fit(
    cov_type='HAC',
    cov_kwds={
        'maxlags': 168
    }
)

In [74]:
hac_sensitivity = pd.DataFrame({
    'coef':
        hourly_model.params[
            weather_terms
        ],

    'p_HAC24':
        hourly_model.pvalues[
            weather_terms
        ],

    'p_HAC48':
        hourly_model_48.pvalues[
            weather_terms
        ],

    'p_HAC168':
        hourly_model_168.pvalues[
            weather_terms
        ]
})

hac_sensitivity.round(4)

,coef,p_HAC24,p_HAC48,p_HAC168
z_temperature,-0.0018,0.9014,0.9204,0.9381
z_humidity,-0.0010,0.9405,0.9532,0.9629
z_wind_speed,-0.0183,0.0003,0.0023,0.0081
z_log_rain,-0.0439,0.0000,0.0001,0.0005
z_pressure,0.0434,0.0000,0.0000,0.0004
wind_dir_sin,-0.0584,0.0000,0.0000,0.0000
wind_dir_cos,-0.0011,0.8703,0.8820,0.8911
z_pm25_lag1,0.5225,0.0000,0.0000,0.0000


In [75]:
daily_model_14 = smf.ols(
    daily_formula,
    daily_model_df
).fit(
    cov_type='HAC',
    cov_kwds={'maxlags': 14}
)

daily_model_30 = smf.ols(
    daily_formula,
    daily_model_df
).fit(
    cov_type='HAC',
    cov_kwds={'maxlags': 30}
)

In [76]:
# HAC seasonal regression as a robustness check
season_hac_df = daily.copy()

season_hac_df['log_pm25'] = np.log1p(
    season_hac_df['pm25_mean']
)

season_hac_df['winter'] = (
    season_hac_df['season'] == 'Winter'
).astype(int)

season_hac_df['pre_monsoon'] = (
    season_hac_df['season'] == 'Pre-Monsoon'
).astype(int)

season_hac_df['post_monsoon'] = (
    season_hac_df['season'] == 'Post-Monsoon'
).astype(int)

In [77]:
season_hac_model = smf.ols(
    """
    log_pm25 ~
    winter +
    pre_monsoon +
    post_monsoon
    """,
    data=season_hac_df
).fit(
    cov_type='HAC',
    cov_kwds={
        'maxlags': 30
    }
)

print(
    season_hac_model.summary()
)

                            OLS Regression Results                            
Dep. Variable:               log_pm25   R-squared:                       0.389
Model:                            OLS   Adj. R-squared:                  0.387
Method:                 Least Squares   F-statistic:                     21.61
Date:                Sun, 27 Sep 2026   Prob (F-statistic):           1.49e-13
Time:                        23:03:10   Log-Likelihood:                -569.53
No. Observations:                 973   AIC:                             1147.
Df Residuals:                     969   BIC:                             1167.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        2.9547      0.073     40.409   

In [78]:
season_hac_model.t_test(
    'pre_monsoon - post_monsoon = 0'
)

<class 'statsmodels.stats.contrast.ContrastResults'>
                             Test for Constraints                             
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.0236      0.146      0.162      0.872      -0.262       0.309